In [3]:
import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import os

print("Loading data...")
train_df = pd.read_csv("../data/ats_scores/train.csv")

label_mapping = {
    'Good Fit': 1.0, 'Potential Fit': 0.7, 'Maybe': 0.5, 'No Fit': 0.1, 'Bad Fit': 0.0
}
def clean_label(value):
    if isinstance(value, (int, float)): return float(value)
    return label_mapping.get(str(value).strip(), 0.5)

# --- THE FIX: The Gold Standard Anchor ---
ideal_profile = "Highly qualified candidate. Excellent technical skills, perfect match for the job requirements, extensive experience, and strong educational background."

train_examples = []
for idx, row in train_df.iterrows():
    train_examples.append(InputExample(
        texts=[str(row['text']), ideal_profile], # Compare resume to the IDEAL profile
        label=clean_label(row['ats_score'])
    ))

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

print("Loading base model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
train_loss = losses.CosineSimilarityLoss(model)

print("Starting True Fine-Tuning against Gold Standard...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)], 
    epochs=4,
    warmup_steps=200,
    show_progress_bar=True
)

save_path = "../app/core/models/ats_scorer_model"
os.makedirs(save_path, exist_ok=True)
model.save(save_path)

print(f"Success! Model saved.")

Loading data...
Loading base model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting True Fine-Tuning against Gold Standard...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/opt/homebrew/Caskroom/miniforge/base/envs/ats_scorer/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,2750.552250
1000,2767.447750


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Success! Model saved.
